# Davinci — Instruction, Reasoning & Agent-Trajectory Data

A schema-first Google Colab pipeline for converting reviewed Persian / bilingual instruction datasets and agent traces into a consistent training format.

**Outputs:** canonical chat SFT JSONL, tool-trajectory JSONL, validation report, source/license manifest.

**Safety against silent corruption:** dataset columns are inspected, mappings are explicit, and unsupported structures are rejected rather than guessed. This notebook does not invent tool traces or certify reasoning correctness.

## 1. Install and initialize

In [ ]:
!pip -q install -U "datasets>=3.0.0" "huggingface_hub>=0.25.0" "pandas>=2.0.0" "jsonschema>=4.0.0"

In [ ]:
import json, re, hashlib, random, unicodedata
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from tqdm.auto import tqdm

SEED=42
random.seed(SEED)
from google.colab import drive
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Davinci/sft_data")
(ROOT/"raw").mkdir(parents=True,exist_ok=True)
(ROOT/"processed").mkdir(parents=True,exist_ok=True)


## 2. Candidate sources

Inspect each dataset card and upstream terms. These are discovery candidates only; no license is assumed. The Persian SFT aggregate may itself contain components with different provenance and restrictions.

In [ ]:
SOURCES = {
 "farsinstruct": "ParsiAI/FarsInstruct",
 "persian_sft_102k": "amiralifirouzi/Persian-SFT-102K",
 "aya": "CohereForAI/aya_dataset",
 "xp3": "bigscience/xP3",
 "persian_reasoning": "hosseinhimself/persian-alpaca-reasoning-v1",
 "toolbench": "openbmb/ToolBench", # verify current availability/card before use
}
for k,v in SOURCES.items(): print(k, "=>", v)

## 3. Select a source and audit its metadata

In [ ]:
api=HfApi()
SOURCE_KEY="farsinstruct"
DATASET_ID=SOURCES[SOURCE_KEY]
info=api.dataset_info(DATASET_ID)
card=info.card_data.to_dict() if info.card_data else {}
metadata={
 "dataset_id":DATASET_ID,
 "revision":getattr(info,"sha",None),
 "card_url":f"https://huggingface.co/datasets/{DATASET_ID}",
 "license_declared":card.get("license"),
 "language":card.get("language"),
 "last_modified":str(info.last_modified),
}
print(json.dumps(metadata,ensure_ascii=False,indent=2))
if not metadata["revision"]:
    raise RuntimeError("No immutable dataset revision was returned; stop rather than stream a moving dataset reference.")
LICENSE_REVIEWED=False
LICENSE_APPROVED_FOR_USE=False
if LICENSE_APPROVED_FOR_USE and not LICENSE_REVIEWED:
    raise ValueError("Cannot approve a dataset before recording a license review.")
LICENSE_NOTES="Review dataset card, upstream components, commercial-use and redistribution restrictions before use."

## 4. Load schema; do not infer field names

In [ ]:
CONFIG=None
SPLIT="train"
kwargs={"split":SPLIT,"streaming":True,"revision":metadata["revision"]}
if CONFIG is not None: kwargs["name"]=CONFIG
stream=load_dataset(DATASET_ID,**kwargs)
print("Features:",stream.features)
example=next(iter(stream.take(1)))
print(json.dumps(example,ensure_ascii=False,indent=2,default=str)[:6000])

## 5. Explicitly map source fields

Choose exactly one adapter after inspecting the example. For simple instruction datasets, map prompt/instruction and response/output. For conversation datasets, write a dedicated adapter that preserves message order and roles. Do not concatenate fields blindly.

In [ ]:
# EDIT THESE after inspecting the schema. Leave unset fields as None.
PROMPT_COLUMN=None       # e.g. "instruction" or "question"
RESPONSE_COLUMN=None     # e.g. "output" or "answer"
SYSTEM_COLUMN=None       # optional system prompt column

MAX_ROWS=20000
if PROMPT_COLUMN is None or RESPONSE_COLUMN is None:
    raise ValueError("Set PROMPT_COLUMN and RESPONSE_COLUMN from the inspected schema.")

stream=load_dataset(DATASET_ID,**kwargs)
records=[]
for idx,row in enumerate(tqdm(stream)):
    if idx>=MAX_ROWS: break
    prompt=row.get(PROMPT_COLUMN)
    answer=row.get(RESPONSE_COLUMN)
    system=row.get(SYSTEM_COLUMN) if SYSTEM_COLUMN else None
    if isinstance(prompt,str) and isinstance(answer,str) and prompt.strip() and answer.strip():
        messages=[]
        if isinstance(system,str) and system.strip(): messages.append({"role":"system","content":system.strip()})
        messages += [{"role":"user","content":prompt.strip()},{"role":"assistant","content":answer.strip()}]
        records.append({"source":DATASET_ID,"source_row":idx,"messages":messages})
print("Mapped records:",len(records))

## 6. Canonical chat schema

Each training example is a sequence of role/content messages. Tool-call examples are represented separately below; this simple SFT adapter intentionally handles text-only prompt/response pairs.

In [ ]:
ALLOWED_ROLES={"system","user","assistant"}
def validate_chat(rec):
    msgs=rec.get("messages")
    if not isinstance(msgs,list) or len(msgs)<2: return False,"messages_missing_or_short"
    for m in msgs:
        if not isinstance(m,dict): return False,"message_not_object"
        if m.get("role") not in ALLOWED_ROLES: return False,"invalid_role"
        if not isinstance(m.get("content"),str): return False,"content_not_string"
        if not m["content"].strip(): return False,"empty_content"
    if not any(m["role"]=="user" for m in msgs): return False,"no_user_message"
    if not any(m["role"]=="assistant" for m in msgs): return False,"no_assistant_message"
    return True,None
valid_chat=[]; invalid_chat=[]; seen=set()
for rec in records:
    ok,reason=validate_chat(rec)
    if not ok: invalid_chat.append({"source_row":rec["source_row"],"reason":reason}); continue
    key=hashlib.sha256(json.dumps(rec["messages"],ensure_ascii=False,sort_keys=True).encode()).hexdigest()
    if key in seen: invalid_chat.append({"source_row":rec["source_row"],"reason":"exact_duplicate"}); continue
    seen.add(key); valid_chat.append({**rec,"id":key})
print("valid",len(valid_chat),"invalid",len(invalid_chat))

## 7. Optional Persian / English audit

Script ratio is a weak diagnostic only. Do not use it as a definitive language classifier or automatically discard code-mixed examples.

In [ ]:
FA_RE=re.compile(r"[\u0600-\u06FF]")
LATIN_RE=re.compile(r"[A-Za-z]")
def script_stats(rec):
    text=" ".join(m["content"] for m in rec["messages"])
    return {"persian_chars":len(FA_RE.findall(text)),"latin_chars":len(LATIN_RE.findall(text)),"chars":len(text)}
if valid_chat:
    stats=pd.DataFrame([script_stats(r) for r in valid_chat])
    display(stats.describe())

## 8. Reasoning-data handling

Reasoning corpora may contain model-generated explanations, incorrect derivations, or hidden prompt artifacts. Keep source provenance, inspect a manually sampled set, and verify math/code answers with deterministic tools where possible. Do not treat a field named "reasoning" as proof of correctness.

In [ ]:
REASONING_REVIEWED=False
REASONING_REVIEW_NOTES="Not reviewed"
# Optional manual sample for review
for r in valid_chat[:5]:
    print("\n--- sample ---")
    for m in r["messages"]: print(m["role"]+":",m["content"][:1200])

## 9. Agent/tool trajectory schema

Use this schema only for real, recorded trajectories or carefully validated demonstrations. An assistant tool call must specify a tool name and JSON arguments; observations must be attributed to the tool. Never fabricate successful observations. This notebook provides validation and a template, not a tool execution environment.

In [ ]:
TOOL_TRACE_SCHEMA={
 "trajectory_id":"unique-string",
 "messages":[
   {"role":"user","content":"user request"},
   {"role":"assistant","content":"brief plan or tool rationale"},
   {"role":"assistant","tool_call":{"name":"search","arguments":{"query":"..."} }},
   {"role":"tool","name":"search","content":"actual tool result","trusted":False},
   {"role":"assistant","content":"final answer"}
 ],
 "metadata":{"source":"recorded-runtime","verified":False,"license_review":"pending"}
}
print(json.dumps(TOOL_TRACE_SCHEMA,ensure_ascii=False,indent=2))

## 10. Add your real trajectory file

Upload JSONL exported by your own agent runtime. Expected format: one JSON object per line with `trajectory_id` and `messages`. Tool observations must be genuine recorded outputs. You can skip this section if you do not yet have traces.

In [ ]:
from google.colab import files
uploaded=files.upload()  # choose a .jsonl file, if available
trace_path=None
for name in uploaded:
    if name.endswith(".jsonl"):
        trace_path=Path(name)
        break
print("Trace file:",trace_path)

## 11. Validate tool traces

In [ ]:
def validate_trace(obj):
    if not isinstance(obj,dict) or not isinstance(obj.get("trajectory_id"),str): return False,"missing_trajectory_id"
    msgs=obj.get("messages")
    if not isinstance(msgs,list) or len(msgs)<2: return False,"invalid_messages"
    if not isinstance(obj.get("metadata"),dict): return False,"metadata_required"
    for m in msgs:
        if not isinstance(m,dict): return False,"message_not_object"
        role=m.get("role")
        if role not in {"system","user","assistant","tool"}: return False,"invalid_role"
        if role=="assistant" and "tool_call" in m:
            call=m["tool_call"]
            if not isinstance(call,dict) or not isinstance(call.get("name"),str): return False,"invalid_tool_call"
            if not isinstance(call.get("arguments"),dict): return False,"tool_arguments_must_be_object"
        elif role=="tool":
            if not isinstance(m.get("name"),str) or not isinstance(m.get("content"),str): return False,"invalid_tool_observation"
            if not isinstance(m.get("trusted"),bool): return False,"tool_result_trust_flag_required"
        elif not isinstance(m.get("content"),str): return False,"content_not_string"
    return True,None

traces=[]; bad_traces=[]
if trace_path:
    for line_no,line in enumerate(trace_path.read_text(encoding="utf-8").splitlines(),1):
        try: obj=json.loads(line)
        except Exception: bad_traces.append({"line":line_no,"reason":"invalid_json"}); continue
        ok,why=validate_trace(obj)
        if ok: traces.append(obj)
        else: bad_traces.append({"line":line_no,"reason":why})
print("valid traces:",len(traces),"invalid:",len(bad_traces))

## 12. Export canonical datasets and provenance manifest

In [ ]:
now=datetime.now(timezone.utc).isoformat()
chat_out=ROOT/"processed"/f"{SOURCE_KEY}_chat_sft.jsonl"
trace_out=ROOT/"processed"/"agent_tool_traces.jsonl"
report_out=ROOT/"processed"/"validation_report.json"
manifest_out=ROOT/"processed"/f"{SOURCE_KEY}_manifest.json"

SFT_VALIDATION_FRACTION=0.05
def assign_chat_split(record):
    prompt="\n".join(m["content"] for m in record["messages"] if m["role"]=="user")
    prompt=unicodedata.normalize("NFC",prompt).translate(str.maketrans({"ي":"ی","ى":"ی","ك":"ک"}))
    prompt=prompt.replace("\u0640","")
    prompt=re.sub(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]","",prompt)
    prompt="\n".join(re.sub(r"[ \t]+"," ",line).strip() for line in prompt.split("\n"))
    digest=hashlib.sha256(prompt.encode("utf-8")).hexdigest()
    bucket=int(hashlib.sha256(f"{SEED}:{digest}".encode("ascii")).hexdigest()[:16],16)
    return "validation" if bucket<int(SFT_VALIDATION_FRACTION*(1<<64)) else "train"
for record in valid_chat:
    record["split"]=assign_chat_split(record)
train_chat=[record for record in valid_chat if record["split"]=="train"]
validation_chat=[record for record in valid_chat if record["split"]=="validation"]
if not train_chat or not validation_chat:
    raise ValueError("SFT train/validation split is empty; collect more examples or adjust SFT_VALIDATION_FRACTION.")

def write_jsonl(path,items):
    with path.open("w",encoding="utf-8") as f:
        for item in items: f.write(json.dumps(item,ensure_ascii=False)+"\n")
def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b""):
            digest.update(chunk)
    return digest.hexdigest()
write_jsonl(chat_out,valid_chat)
write_jsonl(trace_out,traces)
report={"chat_valid":len(valid_chat),"chat_invalid":len(invalid_chat),"trace_valid":len(traces),"trace_invalid":len(bad_traces),"chat_rejections":invalid_chat[:1000],"trace_rejections":bad_traces[:1000]}
report_out.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
manifest={
 "created_at_utc":now,"dataset_id":DATASET_ID,"dataset_revision":metadata["revision"],"card_url":metadata["card_url"],
 "purpose":"instruction",
 "license_declared":metadata["license_declared"],
 "license_reviewed":LICENSE_REVIEWED,"license_reviewed_by_user":LICENSE_REVIEWED,
 "license_approved_for_use":LICENSE_APPROVED_FOR_USE,
 "license_notes":LICENSE_NOTES,"license_review_notes":LICENSE_NOTES,"config":CONFIG,"split":SPLIT,
 "prompt_column":PROMPT_COLUMN,"response_column":RESPONSE_COLUMN,"system_column":SYSTEM_COLUMN,
 "rows_read":len(records),"valid_chat_examples":len(valid_chat),
 "train_chat_examples":len(train_chat),"validation_chat_examples":len(validation_chat),
 "validation_fraction":SFT_VALIDATION_FRACTION,
 "split_method":"SHA-256(seed:normalized_user_prompt_sha256) threshold; identical prompts stay together",
 "package_versions":{name:__import__("importlib.metadata",fromlist=["version"]).version(name) for name in ["datasets","huggingface_hub","pandas","tqdm"]},
 "deduplication":"SHA-256 over canonical messages; exact duplicates only",
 "reasoning_reviewed":REASONING_REVIEWED,"reasoning_review_notes":REASONING_REVIEW_NOTES,
 "output_chat":str(chat_out),"output_tool_traces":str(trace_out),
 "chat_jsonl_sha256":sha256_file(chat_out),"tool_traces_jsonl_sha256":sha256_file(trace_out)}
manifest_out.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print(chat_out,trace_out,report_out,manifest_out,sep="\n")

## 13. Download outputs

In [ ]:
for p in [chat_out,trace_out,report_out,manifest_out]: files.download(str(p))

## 14. Training-data quality gates before model training

- Manually review a stratified sample from every source and language category.
- Verify licenses and upstream terms; unresolved means do not include in a public/commercial release.
- Remove personal data and secrets; review privacy and sensitive-data exposure.
- Detect benchmark contamination and duplicated answers across splits.
- Validate tool names and argument schemas against the actual tool registry used by Davinci.
- Verify tool observations and outcomes; separate successful, failed, and recovered trajectories.
- Keep train/validation/test splits grouped by source/document/conversation to prevent leakage.
- Track token counts using Davinci's eventual tokenizer, not estimates from another tokenizer.
- Evaluate Persian fluency, factuality, instruction following, tool selection, argument validity, and refusal behavior independently.

**Next stage:** merge reviewed corpora, create leakage-resistant splits, and build a reproducible training/evaluation configuration. No training run is started by this notebook.